In [1]:
import mujoco
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from env import UnitreeA1Env
from pathlib import Path
import subprocess
import sys


In [2]:
xmlPath = r"..\external\mujoco_menagerie\unitree_a1\scene.xml"
modelsPath = r"..\Models"
if not Path(modelsPath).exists():
	Path(modelsPath).mkdir(parents=True, exist_ok=True)

model = mujoco.MjModel.from_xml_path(xmlPath)
data = mujoco.MjData(model)

dt = float(model.opt.timestep)

In [3]:
import multiprocessing

# See how many cores you have
print(f"Available cores: {multiprocessing.cpu_count()}")

# Rule of thumb: n_envs = number of physical cores
# Leave 1-2 cores free for the OS and main training thread
N_ENVS = 8
print(f"Using {N_ENVS} envs")

Available cores: 16
Using 8 envs


In [ ]:
version = "2.1"

In [5]:
# check if model version already exists
if Path(f"{modelsPath}/a1_walk_v{version}.zip").exists():
	response = input(f"Model [a1_walk_v{version}.zip] already exists. Overwrite? (y/n): ")
	if response.lower() != "y":
		print("Aborting training.")
		exit()


def make_env(xml):
	def _init():
		return Monitor(UnitreeA1Env(xml, max_episode_steps=500000))
	return _init

class LogCallback(BaseCallback):
	def _on_step(self):
		if len(self.model.ep_info_buffer) > 0 and self.n_calls % 10_000 == 0:
			mean_reward = np.mean([e["r"] for e in self.model.ep_info_buffer])
			mean_len    = np.mean([e["l"] for e in self.model.ep_info_buffer])
			print(f"steps={self.num_timesteps:>8} | mean_ep_reward={mean_reward:>8.2f} | mean_ep_len={mean_len:>6.0f}")
		return True

if __name__ == "__main__":
	n_envs = N_ENVS or multiprocessing.cpu_count() - 2
	print(f"Using {n_envs} envs for training")

	env = SubprocVecEnv([make_env(xmlPath) for _ in range(N_ENVS)])

	model = PPO(
		"MlpPolicy",
		env,
		n_steps=2048,
		batch_size=64 * n_envs,
		n_epochs=10,
		gamma=0.99,
		gae_lambda=0.95,
		clip_range=0.2,
		ent_coef=0.01,
		learning_rate=3e-4,
		verbose=0,
		tensorboard_log="./tb_logs/",
		policy_kwargs=dict(
			net_arch=[256, 256]  # bigger network than default [64, 64]
		)
	)

	try:
		model.learn(
			total_timesteps=30_000_000,
			callback=LogCallback(),
			tb_log_name=f"a1_walk_v{version}",
			progress_bar=True,
			reset_num_timesteps = not Path(f"{modelsPath}/a1_walk_v{version}.zip").exists()
		)
	except KeyboardInterrupt:
		print("Training interrupted by user. Saving model...")
	except Exception as e:
		print(f"An error occurred: {e}")
	finally:
		model.save(f"{modelsPath}/a1_walk_v{version}.zip")
		env.close()
		print(f"Model saved to {Path(f'{modelsPath}/a1_walk_v{version}.zip').absolute()}")

Using 8 envs for training


Output()

steps=   80000 | mean_ep_reward=  163.21 | mean_ep_len=   255

steps=  160000 | mean_ep_reward=  427.30 | mean_ep_len=   324

steps=  240000 | mean_ep_reward=  530.35 | mean_ep_len=   337

...

steps=29680000 | mean_ep_reward=  306.87 | mean_ep_len=   286

steps=29760000 | mean_ep_reward=  220.46 | mean_ep_len=   271

steps=29840000 | mean_ep_reward=  257.03 | mean_ep_len=   296

steps=29920000 | mean_ep_reward=  220.55 | mean_ep_len=   275

steps=30000000 | mean_ep_reward=  237.71 | mean_ep_len=   277

tensorboard --logdir ./lab/tb_logs/

In [7]:
from stable_baselines3 import PPO
import numpy as np
import time

speed_multiplier = 0.5  # Adjust this to speed up or slow down the simulation

# Load the trained model
model = PPO.load(r"..\Models\a1_walk_v2-1")

# Create a render env
env = UnitreeA1Env(xmlPath, render_mode="human", max_episode_steps=10000)
dt = float(env.model.opt.timestep)
obs, _ = env.reset()

while True:
	action, _ = model.predict(obs, deterministic=True)  # deterministic=True = no random sampling
	obs, reward, terminated, truncated, _ = env.step(action)
	env.render()
	time.sleep(dt / speed_multiplier)
	if terminated or truncated:
		obs, _ = env.reset()

KeyboardInterrupt: 